# <span style="color: #1F1DB5;">SPRINT 16 – Series Temporales

# <span style="color: #1F1DB5;">Notebook 2 – Modelos ARIMA y Predicción Temporal — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Comprender la estructura del modelo ARIMA(p, d, q) y qué representa cada parámetro.
- Usar autoarima para selección automática de parámetros óptimos.
- Aplicar SARIMA para series con componente estacional.
- Realizar split temporal correcto (¡nunca aleatorio en series de tiempo!).
- Evaluar predicciones con RMSE, MAE y MAPE.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender la estructura del modelo **ARIMA(p, d, q)** y qué representa cada parámetro.
- Usar **auto_arima** para selección automática de parámetros óptimos.
- Aplicar **SARIMA** para series con componente estacional.
- Realizar **split temporal** correcto (¡nunca aleatorio en series de tiempo!).
- Evaluar predicciones con **RMSE, MAE y MAPE**.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
ARIMA: estructura y parámetros | 25 min |
auto_arima y selección de modelo | 20 min |
SARIMA para series estacionales | 15 min |
Evaluación y split temporal | 15 min |
Pair Programming | 20 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo elegir el modelo correcto para predecir una serie de tiempo?

En la clase anterior aprendimos a descomponer y entender una serie. Hoy damos el paso a la **predicción**: ¿cómo usamos esa comprensión para construir un modelo que haga pronósticos confiables? La familia ARIMA es la herramienta clásica más poderosa para esto.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">ARIMA: Estructura y Parámetros (25 mins)

**ARIMA** significa **A**uto**R**egressive **I**ntegrated **M**oving **A**verage. Combina tres ideas:

**AR (AutoRegressive) – parámetro p:**
- El valor actual depende de los **p valores anteriores**.
- Como una regresión donde las features son los lags de la misma serie.
- Ejemplo: "Las ventas de hoy dependen de las ventas de ayer y anteayer" → AR(2).

**I (Integrated) – parámetro d:**
- Número de **diferenciaciones** necesarias para hacer la serie estacionaria.
- d=1 significa que diferenciamos una vez: `y_diff = y(t) - y(t-1)`.
- En la clase anterior, el test ADF nos dice si necesitamos diferenciar.

**MA (Moving Average) – parámetro q:**
- El valor actual depende de los **q errores de predicción anteriores**.
- "Mi predicción de hoy se ajusta por cuánto me equivoqué ayer".

**ARIMA(p, d, q)** combina las tres ideas. Ejemplos:
- ARIMA(1,1,0) = AR(1) con una diferenciación.
- ARIMA(0,1,1) = MA(1) con una diferenciación.
- ARIMA(1,1,1) = Combina AR y MA con diferenciación.

**¿Cómo elegir p, d, q?**
- **d:** Determinado por el test ADF (cuántas diferenciaciones necesitas).
- **p:** Sugerido por PACF (dónde se corta).
- **q:** Sugerido por ACF (dónde se corta).

Vamos a construir nuestra serie de datos y aplicar un modelo ARIMA paso a paso. Primero creamos los datos, luego determinamos los parámetros y finalmente ajustamos el modelo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Serie tipo AirPassengers
np.random.seed(42)
n = 144
dates = pd.date_range('1949-01-01', periods=n, freq='MS')
trend = np.linspace(100, 500, n)
seasonal = 40 * np.sin(2 * np.pi * np.arange(n) / 12)
noise = np.random.normal(0, 15, n)
passengers = trend * (1 + seasonal / 200) + noise
ts = pd.Series(passengers, index=dates, name='Pasajeros')

# Paso 1: Determinar d con test ADF
result = adfuller(ts, autolag='AIC')
print(f"Serie original - p-value ADF: {result[1]:.6f} → {'Estacionaria' if result[1] < 0.05 else 'NO estacionaria'}")

ts_diff = ts.diff().dropna()
result_diff = adfuller(ts_diff, autolag='AIC')
print(f"Diferenciada 1 vez - p-value ADF: {result_diff[1]:.6f} → {'Estacionaria' if result_diff[1] < 0.05 else 'NO estacionaria'}")
print(f"\n→ Necesitamos d=1")

# Paso 2: Determinar p y q con PACF y ACF
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts_diff, lags=30, ax=axes[0], title='ACF (sugiere q)')
plot_pacf(ts_diff, lags=30, ax=axes[1], title='PACF (sugiere p)')
plt.tight_layout()
plt.show()
print("Observa dónde se cortan las gráficas para elegir p y q.")

## <span style="color: #2826DE;">auto_arima y Selección de Modelo (20 mins)

Elegir p, d, q manualmente mirando ACF/PACF funciona, pero puede ser subjetivo. **auto_arima** de la librería `pmdarima` automatiza esta búsqueda:

1. Prueba múltiples combinaciones de (p, d, q).
2. Usa el criterio **AIC** (Akaike Information Criterion) para comparar.
3. AIC menor = mejor modelo (balancea ajuste vs. complejidad).

**Ventajas:**
- Rápido y objetivo.
- Prueba estacionalidad automáticamente si se lo pides.

**Cuidado:**
- No siempre el modelo con menor AIC es el mejor para predicción.
- Siempre valida con datos de prueba reales.

Usemos auto_arima para encontrar el mejor modelo automáticamente y comparemos con nuestra selección manual.

In [ ]:
from pmdarima import auto_arima

# auto_arima busca los mejores parámetros
model_auto = auto_arima(
    ts,
    start_p=0, max_p=5,
    start_q=0, max_q=5,
    d=None,           # Deja que auto_arima determine d
    seasonal=False,   # Por ahora sin estacionalidad
    trace=True,       # Muestra la búsqueda
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print(f"\nMejor modelo encontrado: {model_auto.order}")
print(f"AIC: {model_auto.aic():.2f}")
print(model_auto.summary())

Preguntas:

- ¿Por qué el AIC penaliza la complejidad del modelo además del error?

- ¿Qué pasaría si usáramos un ARIMA(10,2,10)? ¿Sería mejor porque tiene más parámetros?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">SARIMA para Series Estacionales (15 mins)

Nuestra serie de pasajeros tiene **estacionalidad** clara (patrón que se repite cada 12 meses). ARIMA básico no captura esto bien. Necesitamos **SARIMA**:

**SARIMA(p, d, q)(P, D, Q, m)** donde:
- (p, d, q) = componentes NO estacionales (como antes).
- (P, D, Q) = componentes estacionales (misma lógica pero para el ciclo estacional).
- m = periodo estacional (12 para datos mensuales, 4 para trimestrales, 7 para diarios con ciclo semanal).

**Ejemplo:** SARIMA(1,1,1)(1,1,1,12) significa:
- AR(1), diferenciación 1, MA(1) para la parte no estacional.
- AR estacional(1), diferenciación estacional(1), MA estacional(1) con periodo 12.

Usemos auto_arima con el parámetro `seasonal=True` para encontrar el mejor SARIMA y veamos cómo mejora respecto al ARIMA simple.

In [ ]:
# SARIMA con auto_arima
model_sarima = auto_arima(
    ts,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=None,
    seasonal=True,    # ¡Ahora sí buscamos estacionalidad!
    m=12,             # Periodo estacional = 12 meses
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    D=None,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print(f"\nMejor SARIMA: {model_sarima.order} x {model_sarima.seasonal_order}")
print(f"AIC ARIMA simple: {model_auto.aic():.2f}")
print(f"AIC SARIMA: {model_sarima.aic():.2f}")
print(f"\n→ {'SARIMA' if model_sarima.aic() < model_auto.aic() else 'ARIMA'} es mejor (menor AIC)")

## <span style="color: #2826DE;">Evaluación y Split Temporal (15 mins)

**⚠️ REGLA DE ORO en series de tiempo: NUNCA hagas split aleatorio.**

¿Por qué? Porque en una serie temporal, el futuro depende del pasado. Si mezclas datos futuros en el entrenamiento, estás haciendo **data leakage temporal**: tu modelo "ve el futuro" durante el entrenamiento.

**Split correcto:**
- Train = primeros N% de los datos (pasado).
- Test = últimos (100-N)% de los datos (futuro).
- Típicamente: 80% train, 20% test.

**Métricas de evaluación:**
- **RMSE** (Root Mean Squared Error): Penaliza errores grandes. En las mismas unidades que los datos.
- **MAE** (Mean Absolute Error): Error promedio absoluto. Más robusto a outliers.
- **MAPE** (Mean Absolute Percentage Error): Error como porcentaje. Fácil de interpretar ("error del 5%").

Hagamos el split temporal correcto, entrenemos nuestro SARIMA en el pasado y evaluemos en el futuro. Esto simula lo que pasaría en producción.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Split temporal: 80% train, 20% test
split_point = int(len(ts) * 0.8)
train = ts[:split_point]
test = ts[split_point:]

print(f"Train: {train.index[0].date()} a {train.index[-1].date()} ({len(train)} obs)")
print(f"Test:  {test.index[0].date()} a {test.index[-1].date()} ({len(test)} obs)")

# Entrenar SARIMA solo con datos de train
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12))
fitted = model.fit(disp=False)

# Predicción sobre periodo de test
forecast = fitted.forecast(steps=len(test))

# Métricas
rmse = np.sqrt(mean_squared_error(test, forecast))
mae = mean_absolute_error(test, forecast)
mape = np.mean(np.abs((test - forecast) / test)) * 100

print(f"\n--- Evaluación ---")
print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"MAPE: {mape:.2f}%")

# Visualización
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train, label='Train', color='steelblue')
ax.plot(test, label='Test (real)', color='green')
ax.plot(test.index, forecast, label='Predicción', color='red', linestyle='--')
ax.axvline(test.index[0], color='black', linestyle=':', alpha=0.5)
ax.legend()
ax.set_title(f'SARIMA - Predicción vs Realidad (MAPE: {mape:.1f}%)')
plt.tight_layout()
plt.show()

Preguntas:

- ¿Por qué un split aleatorio sería un error grave en series de tiempo?

- ¿Cuándo preferirías usar MAPE sobre RMSE para evaluar tu modelo?

- ¿Qué harías si el MAPE es del 25%? ¿El modelo es útil o no?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Actividad práctica – Pair Programming (20 mins)

**Dinámica:** Trabajen en parejas. Una persona es el "driver" (escribe código) y la otra es el "navigator" (guía y revisa). Cambien roles a la mitad.

**Ejercicio:** Con la serie de ventas proporcionada:
1. Determinen si es estacionaria (ADF test).
2. Usen `auto_arima` para encontrar el mejor modelo (con estacionalidad).
3. Hagan split temporal 80/20.
4. Entrenen el modelo y hagan forecast.
5. Calculen RMSE, MAE y MAPE.
6. **Bonus:** ¿Mejora si aplican transformación logarítmica antes de modelar?

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: Datos para Pair Programming


## <span style="color: #2826DE;">Tips y buenas prácticas

- **Nunca** hagas train_test_split aleatorio en series de tiempo. Usa corte cronológico.
- `auto_arima` es un excelente punto de partida, pero valida siempre con datos de prueba.
- Si tu serie tiene varianza creciente, aplica `log()` antes de modelar y luego `exp()` para las predicciones.
- MAPE no funciona bien cuando los valores reales son cercanos a cero (divisiones por valores pequeños).
- Siempre grafica predicción vs realidad: las métricas numéricas no cuentan toda la historia.

## <span style="color: #2826DE;">Diagnóstico de Residuos del Modelo

Un buen modelo de series de tiempo debe tener residuos que se comporten como **ruido blanco**: sin patrones, sin autocorrelación y con distribución normal. Si los residuos muestran patrones, significa que el modelo no capturó toda la información disponible.

In [ ]:
# Diagnóstico de residuos del modelo SARIMA
residuals = fitted.resid

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Residuos en el tiempo
axes[0,0].plot(residuals, color='steelblue')
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Residuos en el tiempo')

# Histograma
axes[0,1].hist(residuals, bins=20, color='steelblue', edgecolor='white', density=True)
axes[0,1].set_title('Distribución de residuos')

# QQ-plot
from scipy import stats
stats.probplot(residuals, plot=axes[1,0])
axes[1,0].set_title('QQ-Plot (normalidad)')

# ACF de residuos
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(residuals, lags=20, ax=axes[1,1], title='ACF de residuos')

plt.tight_layout()
plt.show()

print("Si los residuos son ruido blanco:")
print("  ✓ No hay patrones en la gráfica temporal")
print("  ✓ La distribución es aproximadamente normal")
print("  ✓ No hay autocorrelación significativa en ACF")

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué representa el parámetro "d" en ARIMA(p,d,q)?

- Número de componentes autoregresivos
- Número de diferenciaciones para estacionariedad 
- Grado del promedio móvil
- Dimensión del modelo


2️⃣ ¿Cómo debe hacerse el split train/test en series de tiempo?

- Aleatorio con random_state=42
- Estratificado por valores
- Cronológico: pasado entrena, futuro evalúa 
- 50/50 alternando filas


3️⃣ ¿Qué criterio usa auto_arima para elegir el mejor modelo?

- RMSE
- R²
- AIC (Akaike Information Criterion) 
- Accuracy


4️⃣ ¿Qué agrega SARIMA respecto a ARIMA?

- Más datos de entrenamiento
- Componentes estacionales (P, D, Q, m) 
- Mejor optimización numérica
- Datos exógenos


5️⃣ ¿Qué significa un MAPE del 8%?

- El modelo explica el 8% de la varianza
- El error promedio es del 8% respecto a los valores reales 
- Se necesitan 8 iteraciones para converger
- El modelo tiene 8 parámetros


6️⃣ ¿Por qué un split aleatorio es incorrecto en series de tiempo?

- Porque los datos son muy grandes
- Porque causa data leakage temporal 
- Porque sklearn no lo soporta
- Porque ARIMA no acepta datos desordenados